In [3]:
from google.colab import files

uploaded = files.upload()

Saving gangnam_citizen_life_month_features_2025_07_12.csv to gangnam_citizen_life_month_features_2025_07_12.csv
Saving gangnam_db1_clustered_202507_202512.csv to gangnam_db1_clustered_202507_202512 (1).csv


In [6]:
import pandas as pd

# 1. SKT 유동인구
skt = pd.read_csv(
    "gangnam_db1_clustered_202507_202512.csv",
    encoding="cp949",
    sep="\t"
)

# 2. 서울 시민생활 데이터
citizen_life_month_feature = pd.read_csv(
    "gangnam_citizen_life_month_features_2025_07_12.csv",
    encoding="utf-8-sig"
)

print("[SKT]")
print("shape:", skt.shape)

print("\n[서울 시민생활]")
print("shape:", citizen_life_month_feature.shape)

[SKT]
shape: (132, 36)

[서울 시민생활]
shape: (132, 25)


In [7]:
# SKT Key 생성
skt["행정동코드_key"] = skt["adm_cd"].astype(str)

skt["기준연월_key"] = (
    skt["STD_YM"]
    .astype(str)
    .str[:4]
    + "-"
    + skt["STD_YM"]
    .astype(str)
    .str[4:6]
)

# 서울 시민생활 Key 생성
citizen_life_month_feature["행정동코드_key"] = (
    citizen_life_month_feature["행정동코드"].astype(str)
)

citizen_life_month_feature["기준연월_key"] = (
    citizen_life_month_feature["기준연월"].astype(str)
)

# 각 데이터 구조 확인
print("[SKT]")
print("행정동 수:", skt["행정동코드_key"].nunique())
print("월 수:", skt["기준연월_key"].nunique())
print(
    "key 중복:",
    skt.duplicated(
        ["행정동코드_key", "기준연월_key"]
    ).sum()
)

print("\n[서울 시민생활]")
print(
    "행정동 수:",
    citizen_life_month_feature["행정동코드_key"].nunique()
)
print(
    "월 수:",
    citizen_life_month_feature["기준연월_key"].nunique()
)
print(
    "key 중복:",
    citizen_life_month_feature.duplicated(
        ["행정동코드_key", "기준연월_key"]
    ).sum()
)

# 두 데이터의 Key 일치 여부 확인
key_check = (
    skt[
        ["행정동코드_key", "기준연월_key"]
    ]
    .merge(
        citizen_life_month_feature[
            ["행정동코드_key", "기준연월_key"]
        ],
        on=["행정동코드_key", "기준연월_key"],
        how="outer",
        indicator=True
    )
)

print("\n[Key 일치 결과]")
print(key_check["_merge"].value_counts())

[SKT]
행정동 수: 22
월 수: 6
key 중복: 0

[서울 시민생활]
행정동 수: 22
월 수: 6
key 중복: 0

[Key 일치 결과]
_merge
both          132
left_only       0
right_only      0
Name: count, dtype: int64


In [8]:
for i, col in enumerate(skt.columns, 1):
    print(f"{i:2d}. {col}")

 1. adm_cd
 2. adm_nm
 3. dong_name
 4. total_flow
 5. point_count
 6. flow_per_point
 7. male_ratio
 8. female_ratio
 9. age_10_ratio
10. age_20_ratio
11. age_30_ratio
12. age_40_ratio
13. age_50_ratio
14. age_60_plus_ratio
15. time_total
16. time_point_count
17. time_flow_per_point
18. night_ratio
19. morning_ratio
20. daytime_ratio
21. evening_ratio
22. peak_hour
23. peak_hour_ratio
24. week_total
25. weekday_point_count
26. week_flow_per_point
27. workday_ratio
28. weekend_ratio
29. weekend_weekday_index
30. sat_ratio
31. sun_ratio
32. peak_day
33. peak_day_ratio
34. STD_YM
35. cluster_k2
36. cluster_k6
37. 행정동코드_key
38. 기준연월_key


In [9]:
skt_feature_cols = [
    # 규모
    "total_flow",
    "flow_per_point",

    # 성별 구성
    "male_ratio",
    "female_ratio",

    # 연령 구성
    "age_10_ratio",
    "age_20_ratio",
    "age_30_ratio",
    "age_40_ratio",
    "age_50_ratio",
    "age_60_plus_ratio",

    # 시간대 특성
    "night_ratio",
    "morning_ratio",
    "daytime_ratio",
    "evening_ratio",
    "peak_hour",
    "peak_hour_ratio",

    # 요일 특성
    "workday_ratio",
    "weekend_ratio",
    "weekend_weekday_index",
    "sat_ratio",
    "sun_ratio",
    "peak_day",
    "peak_day_ratio"
]

skt_analysis = skt[
    [
        "행정동코드_key",
        "dong_name",
        "기준연월_key"
    ]
    + skt_feature_cols
].copy()

print("SKT 분석용 shape:", skt_analysis.shape)
print("Feature 수:", len(skt_feature_cols))
print("결측값:", skt_analysis.isna().sum().sum())
print(
    "key 중복:",
    skt_analysis.duplicated(
        ["행정동코드_key", "기준연월_key"]
    ).sum()
)

display(skt_analysis.head())

SKT 분석용 shape: (132, 26)
Feature 수: 23
결측값: 0
key 중복: 0


,행정동코드_key,dong_name,기준연월_key,total_flow,flow_per_point,male_ratio,female_ratio,age_10_ratio,age_20_ratio,age_30_ratio,...,evening_ratio,peak_hour,peak_hour_ratio,workday_ratio,weekend_ratio,weekend_weekday_index,sat_ratio,sun_ratio,peak_day,peak_day_ratio
0,1123051,신사동,2025-07,432575.57,618.849170,0.503326,0.496674,0.075857,0.167979,0.198433,...,0.199028,18,0.077067,0.753631,0.246369,0.817272,0.137825,0.108544,금,0.157919
1,1123052,논현1동,2025-07,355919.32,719.028929,0.523535,0.476465,0.045584,0.196861,0.237137,...,0.193453,18,0.078305,0.798194,0.201806,0.632072,0.115970,0.085836,금,0.163926
2,1123053,논현2동,2025-07,417862.51,722.945519,0.520202,0.479798,0.049218,0.177493,0.233524,...,0.175400,18,0.076990,0.813927,0.186073,0.571529,0.106701,0.079372,금,0.164533
3,1123058,삼성1동,2025-07,626441.05,853.461921,0.525791,0.474209,0.072436,0.175119,0.222469,...,0.154562,12,0.082909,0.759416,0.240584,0.792002,0.133053,0.107531,금,0.161369
4,1123059,삼성2동,2025-07,360403.43,716.507813,0.523394,0.476606,0.073137,0.156488,0.212963,...,0.178792,18,0.075877,0.814064,0.185936,0.571013,0.104239,0.081697,목,0.164810


In [10]:
import numpy as np

composition_check = pd.DataFrame({
    "gender_sum": (
        skt_analysis["male_ratio"]
        + skt_analysis["female_ratio"]
    ),

    "age_sum": skt_analysis[
        [
            "age_10_ratio",
            "age_20_ratio",
            "age_30_ratio",
            "age_40_ratio",
            "age_50_ratio",
            "age_60_plus_ratio"
        ]
    ].sum(axis=1),

    "time_sum": skt_analysis[
        [
            "night_ratio",
            "morning_ratio",
            "daytime_ratio",
            "evening_ratio"
        ]
    ].sum(axis=1),

    "work_weekend_sum": (
        skt_analysis["workday_ratio"]
        + skt_analysis["weekend_ratio"]
    ),

    "sat_sun_sum": (
        skt_analysis["sat_ratio"]
        + skt_analysis["sun_ratio"]
    )
})

print(composition_check.describe().round(6))

print("\n[1과의 최대 차이]")
for col in [
    "gender_sum",
    "age_sum",
    "time_sum",
    "work_weekend_sum"
]:
    print(
        col,
        np.abs(
            composition_check[col] - 1
        ).max()
    )

print("\n[weekend_ratio와 sat+sun 차이]")
print(
    np.abs(
        skt_analysis["weekend_ratio"]
        - composition_check["sat_sun_sum"]
    ).describe()
)

       gender_sum  age_sum  time_sum  work_weekend_sum  sat_sun_sum
count       132.0    132.0     132.0             132.0   132.000000
mean          1.0      1.0       1.0               1.0     0.235911
std           0.0      0.0       0.0               0.0     0.034314
min           1.0      1.0       1.0               1.0     0.176704
25%           1.0      1.0       1.0               1.0     0.209016
50%           1.0      1.0       1.0               1.0     0.231781
75%           1.0      1.0       1.0               1.0     0.257326
max           1.0      1.0       1.0               1.0     0.311890

[1과의 최대 차이]
gender_sum 0.0
age_sum 1.000000082740371e-09
time_sum 1.000000082740371e-09
work_weekend_sum 0.0

[weekend_ratio와 sat+sun 차이]
count    1.320000e+02
mean     2.196970e-10
std      4.156186e-10
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.551115e-17
max      1.000000e-09
dtype: float64


In [11]:
skt_ml_cols = [
    # 규모
    "total_flow",
    "flow_per_point",

    # 성별 구성
    "male_ratio",

    # 연령 구성
    "age_20_ratio",
    "age_30_ratio",
    "age_40_ratio",
    "age_50_ratio",
    "age_60_plus_ratio",

    # 시간대 구성
    "morning_ratio",
    "daytime_ratio",
    "evening_ratio",
    "peak_hour_ratio",

    # 요일 특성
    "workday_ratio",
    "weekend_weekday_index",
    "sat_ratio",
    "peak_day_ratio"
]

skt_ml_feature = skt_analysis[
    [
        "행정동코드_key",
        "dong_name",
        "기준연월_key"
    ]
    + skt_ml_cols
].copy()

print("shape:", skt_ml_feature.shape)
print("SKT ML Feature 수:", len(skt_ml_cols))
print("결측값:", skt_ml_feature.isna().sum().sum())
print(
    "key 중복:",
    skt_ml_feature.duplicated(
        ["행정동코드_key", "기준연월_key"]
    ).sum()
)

display(skt_ml_feature.head())

shape: (132, 19)
SKT ML Feature 수: 16
결측값: 0
key 중복: 0


,행정동코드_key,dong_name,기준연월_key,total_flow,flow_per_point,male_ratio,age_20_ratio,age_30_ratio,age_40_ratio,age_50_ratio,age_60_plus_ratio,morning_ratio,daytime_ratio,evening_ratio,peak_hour_ratio,workday_ratio,weekend_weekday_index,sat_ratio,peak_day_ratio
0,1123051,신사동,2025-07,432575.57,618.849170,0.503326,0.167979,0.198433,0.196902,0.171534,0.189296,0.197944,0.557845,0.199028,0.077067,0.753631,0.817272,0.137825,0.157919
1,1123052,논현1동,2025-07,355919.32,719.028929,0.523535,0.196861,0.237137,0.206142,0.156328,0.157949,0.204226,0.552071,0.193453,0.078305,0.798194,0.632072,0.115970,0.163926
2,1123053,논현2동,2025-07,417862.51,722.945519,0.520202,0.177493,0.233524,0.212603,0.162753,0.164409,0.213117,0.570003,0.175400,0.076990,0.813927,0.571529,0.106701,0.164533
3,1123058,삼성1동,2025-07,626441.05,853.461921,0.525791,0.175119,0.222469,0.213145,0.169965,0.146865,0.199860,0.620592,0.154562,0.082909,0.759416,0.792002,0.133053,0.161369
4,1123059,삼성2동,2025-07,360403.43,716.507813,0.523394,0.156488,0.212963,0.216359,0.172404,0.168648,0.218743,0.563141,0.178792,0.075877,0.814064,0.571013,0.104239,0.164810


In [12]:
# ==========================================
# 1. 132행 기준 상관관계
# ==========================================

corr_132 = skt_ml_feature[skt_ml_cols].corr()

pairs_132 = []

for i in range(len(skt_ml_cols)):
    for j in range(i + 1, len(skt_ml_cols)):

        r = corr_132.iloc[i, j]

        if abs(r) >= 0.8:
            pairs_132.append({
                "Feature_1": skt_ml_cols[i],
                "Feature_2": skt_ml_cols[j],
                "상관계수": r
            })

pairs_132 = pd.DataFrame(pairs_132)

if len(pairs_132) > 0:
    pairs_132["절대상관"] = pairs_132["상관계수"].abs()

    pairs_132 = (
        pairs_132
        .sort_values("절대상관", ascending=False)
        .drop(columns="절대상관")
        .reset_index(drop=True)
    )


# ==========================================
# 2. 22개 행정동 6개월 평균
# ==========================================

skt_dong_mean = (
    skt_ml_feature
    .groupby(
        ["행정동코드_key", "dong_name"]
    )[skt_ml_cols]
    .mean()
    .reset_index()
)

corr_22 = skt_dong_mean[skt_ml_cols].corr()

pairs_22 = []

for i in range(len(skt_ml_cols)):
    for j in range(i + 1, len(skt_ml_cols)):

        r = corr_22.iloc[i, j]

        if abs(r) >= 0.8:
            pairs_22.append({
                "Feature_1": skt_ml_cols[i],
                "Feature_2": skt_ml_cols[j],
                "상관계수": r
            })

pairs_22 = pd.DataFrame(pairs_22)

if len(pairs_22) > 0:
    pairs_22["절대상관"] = pairs_22["상관계수"].abs()

    pairs_22 = (
        pairs_22
        .sort_values("절대상관", ascending=False)
        .drop(columns="절대상관")
        .reset_index(drop=True)
    )


# ==========================================
# 3. 결과
# ==========================================

print("[132행 기준]")
print("|r| >= 0.8 관계 수:", len(pairs_132))
display(pairs_132.round(3))

print("\n[22개 동 6개월 평균 기준]")
print("shape:", skt_dong_mean.shape)
print("|r| >= 0.8 관계 수:", len(pairs_22))
display(pairs_22.round(3))

[132행 기준]
|r| >= 0.8 관계 수: 10


,Feature_1,Feature_2,상관계수
0,workday_ratio,weekend_weekday_index,-0.999
1,workday_ratio,sat_ratio,-0.974
2,weekend_weekday_index,sat_ratio,0.971
3,age_20_ratio,age_30_ratio,0.943
4,age_20_ratio,age_50_ratio,-0.862
5,age_60_plus_ratio,morning_ratio,0.862
6,age_60_plus_ratio,daytime_ratio,-0.844
7,age_60_plus_ratio,peak_hour_ratio,-0.825
8,daytime_ratio,peak_hour_ratio,0.821
9,flow_per_point,age_60_plus_ratio,-0.805



[22개 동 6개월 평균 기준]
shape: (22, 18)
|r| >= 0.8 관계 수: 11


,Feature_1,Feature_2,상관계수
0,workday_ratio,weekend_weekday_index,-0.999
1,workday_ratio,sat_ratio,-0.975
2,weekend_weekday_index,sat_ratio,0.972
3,age_20_ratio,age_30_ratio,0.947
4,age_60_plus_ratio,morning_ratio,0.885
5,age_60_plus_ratio,daytime_ratio,-0.870
6,age_20_ratio,age_50_ratio,-0.869
7,age_60_plus_ratio,peak_hour_ratio,-0.846
8,daytime_ratio,peak_hour_ratio,0.832
9,workday_ratio,peak_day_ratio,0.806


In [13]:
skt_core_cols = [
    # 규모
    "total_flow",
    "flow_per_point",

    # 성별
    "male_ratio",

    # 연령
    "age_20_ratio",
    "age_30_ratio",
    "age_40_ratio",
    "age_50_ratio",
    "age_60_plus_ratio",

    # 시간대
    "morning_ratio",
    "daytime_ratio",
    "evening_ratio",
    "peak_hour_ratio",

    # 요일
    "weekend_weekday_index",
    "peak_day_ratio"
]

skt_core_feature = skt_analysis[
    [
        "행정동코드_key",
        "dong_name",
        "기준연월_key"
    ]
    + skt_core_cols
].copy()

print("shape:", skt_core_feature.shape)
print("SKT Core Feature 수:", len(skt_core_cols))
print("결측값:", skt_core_feature.isna().sum().sum())
print(
    "key 중복:",
    skt_core_feature.duplicated(
        ["행정동코드_key", "기준연월_key"]
    ).sum()
)

shape: (132, 17)
SKT Core Feature 수: 14
결측값: 0
key 중복: 0


In [14]:
# 시민생활 분석 Feature
citizen_cols = [
    # 관심집단 9개
    "low_comm_ratio",
    "low_weekday_outing_ratio",
    "low_holiday_outing_ratio",
    "high_worktime_ratio",
    "high_video_ratio",
    "high_life_service_ratio",
    "financial_interest_ratio",
    "very_low_outing_ratio",
    "very_low_outing_comm_ratio",

    # 통신정보 Core 13개
    "야간상주지 변경횟수 평균",
    "평균 근무시간 평균",
    "평일 총 이동 횟수",
    "휴일 총 이동 횟수 평균",
    "집 추정 위치 평일 총 체류시간",
    "집 추정 위치 휴일 총 체류시간",
    "금융 서비스 사용일수",
    "쇼핑 서비스 사용일수",
    "배달 서비스 사용일수",
    "최근 3개월 내 요금 연체 비율",
    "평균 통화대상자 수",
    "데이터 사용량",
    "카카오톡_비사용비율"
]

# 교차상관 검사용 임시 결합
cross_source = (
    skt_core_feature
    .merge(
        citizen_life_month_feature[
            ["행정동코드_key", "기준연월_key"] + citizen_cols
        ],
        on=["행정동코드_key", "기준연월_key"],
        how="inner",
        validate="one_to_one"
    )
)

print("임시 결합 shape:", cross_source.shape)
print("결측값:", cross_source.isna().sum().sum())

임시 결합 shape: (132, 39)
결측값: 0


In [15]:
# ==========================================
# 132행 기준 SKT × 시민생활 교차상관
# ==========================================

cross_corr_132 = (
    cross_source[skt_core_cols + citizen_cols]
    .corr()
    .loc[skt_core_cols, citizen_cols]
)

cross_pairs_132 = []

for skt_var in skt_core_cols:
    for citizen_var in citizen_cols:

        r = cross_corr_132.loc[skt_var, citizen_var]

        if abs(r) >= 0.8:
            cross_pairs_132.append({
                "SKT_Feature": skt_var,
                "시민생활_Feature": citizen_var,
                "상관계수": r
            })

cross_pairs_132 = pd.DataFrame(cross_pairs_132)

if len(cross_pairs_132) > 0:
    cross_pairs_132["절대상관"] = cross_pairs_132["상관계수"].abs()

    cross_pairs_132 = (
        cross_pairs_132
        .sort_values("절대상관", ascending=False)
        .drop(columns="절대상관")
        .reset_index(drop=True)
    )


# ==========================================
# 22개 동 6개월 평균 기준
# ==========================================

cross_source_dong = (
    cross_source
    .groupby("행정동코드_key")[
        skt_core_cols + citizen_cols
    ]
    .mean()
    .reset_index()
)

cross_corr_22 = (
    cross_source_dong[skt_core_cols + citizen_cols]
    .corr()
    .loc[skt_core_cols, citizen_cols]
)

cross_pairs_22 = []

for skt_var in skt_core_cols:
    for citizen_var in citizen_cols:

        r = cross_corr_22.loc[skt_var, citizen_var]

        if abs(r) >= 0.8:
            cross_pairs_22.append({
                "SKT_Feature": skt_var,
                "시민생활_Feature": citizen_var,
                "상관계수": r
            })

cross_pairs_22 = pd.DataFrame(cross_pairs_22)

if len(cross_pairs_22) > 0:
    cross_pairs_22["절대상관"] = cross_pairs_22["상관계수"].abs()

    cross_pairs_22 = (
        cross_pairs_22
        .sort_values("절대상관", ascending=False)
        .drop(columns="절대상관")
        .reset_index(drop=True)
    )

print("[132행 기준]")
print("|r| >= 0.8:", len(cross_pairs_132))
display(cross_pairs_132.round(3))

print("\n[22개 동 평균 기준]")
print("|r| >= 0.8:", len(cross_pairs_22))
display(cross_pairs_22.round(3))

[132행 기준]
|r| >= 0.8: 2


,SKT_Feature,시민생활_Feature,상관계수
0,age_20_ratio,very_low_outing_ratio,0.844
1,age_60_plus_ratio,very_low_outing_ratio,-0.819



[22개 동 평균 기준]
|r| >= 0.8: 5


,SKT_Feature,시민생활_Feature,상관계수
0,age_20_ratio,very_low_outing_ratio,0.875
1,age_60_plus_ratio,very_low_outing_ratio,-0.850
2,age_50_ratio,very_low_outing_ratio,-0.822
3,age_30_ratio,very_low_outing_ratio,0.817
4,daytime_ratio,very_low_outing_ratio,0.814


In [16]:
# ==========================================
# DB1 Dong-Month Mart 생성
# ==========================================

dong_month_mart = (
    skt_core_feature
    .merge(
        citizen_life_month_feature[
            ["행정동코드_key", "기준연월_key"]
            + citizen_cols
        ],
        on=[
            "행정동코드_key",
            "기준연월_key"
        ],
        how="inner",
        validate="one_to_one"
    )
)

# 컬럼명 정리
dong_month_mart = dong_month_mart.rename(
    columns={
        "행정동코드_key": "행정동코드",
        "dong_name": "행정동명",
        "기준연월_key": "기준연월"
    }
)

# ==========================================
# 최종 구조 검증
# ==========================================

print("Dong-Month Mart shape:",
      dong_month_mart.shape)

print(
    "행정동 수:",
    dong_month_mart["행정동코드"].nunique()
)

print(
    "월 수:",
    dong_month_mart["기준연월"].nunique()
)

print(
    "key 중복:",
    dong_month_mart.duplicated(
        ["행정동코드", "기준연월"]
    ).sum()
)

print(
    "결측값:",
    dong_month_mart.isna().sum().sum()
)

print(
    "SKT Feature:",
    len(skt_core_cols)
)

print(
    "시민생활 Feature:",
    len(citizen_cols)
)

display(dong_month_mart.head())

Dong-Month Mart shape: (132, 39)
행정동 수: 22
월 수: 6
key 중복: 0
결측값: 0
SKT Feature: 14
시민생활 Feature: 22


,행정동코드,행정동명,기준연월,total_flow,flow_per_point,male_ratio,age_20_ratio,age_30_ratio,age_40_ratio,age_50_ratio,...,휴일 총 이동 횟수 평균,집 추정 위치 평일 총 체류시간,집 추정 위치 휴일 총 체류시간,금융 서비스 사용일수,쇼핑 서비스 사용일수,배달 서비스 사용일수,최근 3개월 내 요금 연체 비율,평균 통화대상자 수,데이터 사용량,카카오톡_비사용비율
0,1123051,신사동,2025-07,432575.57,618.849170,0.503326,0.167979,0.198433,0.196902,0.171534,...,11.205907,553582.987816,113811.790351,149.557951,217.714911,11.734033,6.776245,13.876814,5.866857,5.425012
1,1123052,논현1동,2025-07,355919.32,719.028929,0.523535,0.196861,0.237137,0.206142,0.156328,...,10.216667,601387.594908,129611.695029,173.788135,240.202800,15.235412,14.922052,14.203254,7.997090,5.218840
2,1123053,논현2동,2025-07,417862.51,722.945519,0.520202,0.177493,0.233524,0.212603,0.162753,...,10.755204,620728.488391,126176.974663,162.222892,230.679331,13.345595,10.151902,14.787257,6.618736,5.604319
3,1123058,삼성1동,2025-07,626441.05,853.461921,0.525791,0.175119,0.222469,0.213145,0.169965,...,11.703325,576179.108701,124279.866563,149.269422,215.785875,11.122984,6.437925,14.410969,5.610517,5.414893
4,1123059,삼성2동,2025-07,360403.43,716.507813,0.523394,0.156488,0.212963,0.216359,0.172404,...,10.756243,611285.226398,124373.174955,169.322581,239.950272,12.622765,8.281708,14.415517,6.327152,4.078160


In [17]:
output_file = "gangnam_db1_dong_month_behavior_mart_2025_07_12.csv"

dong_month_mart.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_file)
print("shape:", dong_month_mart.shape)

저장 완료: gangnam_db1_dong_month_behavior_mart_2025_07_12.csv
shape: (132, 39)


In [18]:
from google.colab import files

files.download(
    "gangnam_db1_dong_month_behavior_mart_2025_07_12.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>